In [1]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/elifsudayi/food-recipe-finder.git
%cd food-recipe-finder
!pip install -q -r requirements.txt
!tar -xf /content/drive/MyDrive/swe015/food10.tar -C /content
!mkdir -p models/onnx results
!cp /content/drive/MyDrive/swe015/vgg16_ft.pt /content/drive/MyDrive/swe015/small_cnn.pt models/
!lscpu | grep -E "Model name|^CPU\(s\)"
!grep -o -E "avx512_vnni|avx2" /proc/cpuinfo | sort -u
import onnxruntime as ort; print("onnxruntime", ort.__version__)

Mounted at /content/drive
Cloning into 'food-recipe-finder'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 93 (delta 34), reused 76 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 491.04 KiB | 8.61 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/food-recipe-finder
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
avx2
onnxruntime 1.32.0


In [2]:
import sys, json; sys.path.append(".")
import numpy as np, torch
from src.image.models import build_vgg16, SmallCNN
from src.image.data import get_loaders
from src.image.train import evaluate
from src.quant.onnx_utils import *
from src.utils import hardware_info, model_size_mb, get_device

vgg = build_vgg16(10, pretrained=False)
vgg.load_state_dict(torch.load("models/vgg16_ft.pt", map_location="cpu")); vgg.eval()
FP32, INT8 = "models/onnx/vgg16_fp32.onnx", "models/onnx/vgg16_int8.onnx"
export_vgg16_onnx(vgg, FP32)
print("fp32 ONNX MB:", round(file_mb(FP32), 1))

test_loader = get_loaders("/content/food10", 224, batch_size=32, splits=("test",))[0]["test"]
x, y = next(iter(test_loader))
with torch.no_grad(): ref = vgg(x).numpy()
clf32 = OnnxClassifier(FP32); out = clf32.logits(x.numpy())
print("max |logit difference|:", np.abs(ref - out).max(), "| same estimation rate:", (ref.argmax(1) == out.argmax(1)).mean())

fp32 ONNX MB: 110.3


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


max |logit difference|: 8.583069e-06 | same estimation rate: 1.0


In [3]:
arrays = calib_tensors("/content/food10", per_class=20)
print(len(arrays), "calibration image", arrays[0].shape)
quantize_vgg16_static(FP32, INT8, arrays)
print(f"fp32 ONNX {file_mb(FP32):.1f} MB  ->  INT8 ONNX {file_mb(INT8):.1f} MB")

200 calibration image (1, 3, 224, 224)
fp32 ONNX 110.3 MB  ->  INT8 ONNX 27.7 MB


In [4]:
clf8 = OnnxClassifier(INT8)
r8 = evaluate_onnx(clf8, test_loader)
dev = get_device()
r32 = evaluate(vgg.to(dev), test_loader, dev)
agree = float((np.array(r32["preds"]) == np.array(r8["preds"])).mean())
print(f"fp32 PyTorch : top1 {r32['top1']:.4f} top5 {r32['top5']:.4f}")
print(f"INT8 ONNX    : top1 {r8['top1']:.4f} top5 {r8['top5']:.4f}")
print(f"Same prediction rate as fp32: {agree:.4f}  (n={r8['n']})")

100%|██████████| 79/79 [11:58<00:00,  9.09s/it]


fp32 PyTorch : top1 0.9208 top5 0.9956
INT8 ONNX    : top1 0.9096 top5 0.9928
Same prediction rate as fp32: 0.9616  (n=2500)


In [5]:
vgg_cpu = build_vgg16(10, pretrained=False)
vgg_cpu.load_state_dict(torch.load("models/vgg16_ft.pt", map_location="cpu")); vgg_cpu.eval()
cnn = SmallCNN(10); cnn.load_state_dict(torch.load("models/small_cnn.pt", map_location="cpu")); cnn.eval()
x224, x128 = torch.randn(1, 3, 224, 224), torch.randn(1, 3, 128, 128)
n224 = x224.numpy()
with torch.no_grad():
    lat = {"small_cnn_fp32_pytorch": time_calls(lambda: cnn(x128)),
           "vgg16_fp32_pytorch":     time_calls(lambda: vgg_cpu(x224)),
           "vgg16_fp32_onnx":        time_calls(lambda: clf32.logits(n224)),
           "vgg16_int8_onnx":        time_calls(lambda: clf8.logits(n224))}
for k, v in lat.items(): print(f"{k:26s} {v['mean_ms']:8.1f} ± {v['std_ms']:.1f} ms")

small_cnn_fp32_pytorch         20.2 ± 2.9 ms
vgg16_fp32_pytorch            614.9 ± 122.2 ms
vgg16_fp32_onnx               472.3 ± 97.0 ms
vgg16_int8_onnx               290.5 ± 63.1 ms


In [6]:
out = {"test": {"vgg16_fp32_pytorch": {k: r32[k] for k in ("top1", "top5")},
                "vgg16_int8_onnx": {k: r8[k] for k in ("top1", "top5")},
                "int8_fp32_prediction_agreement": agree, "n_test": r8["n"]},
       "size_mb": {"small_cnn_fp32_pytorch": model_size_mb(cnn), "vgg16_fp32_pytorch": model_size_mb(vgg_cpu),
                   "vgg16_fp32_onnx": file_mb(FP32), "vgg16_int8_onnx": file_mb(INT8)},
       "latency_cpu_ms": lat, "calibration": "200 train images (20 per class), MinMax, QDQ, per-channel",
       "hardware": hardware_info()}
json.dump(out, open("results/quant_vgg16_metrics.json", "w"), indent=2)
print(json.dumps(out, indent=2))
!cp models/onnx/vgg16_int8.onnx models/onnx/vgg16_fp32.onnx results/quant_vgg16_metrics.json /content/drive/MyDrive/swe015/

{
  "test": {
    "vgg16_fp32_pytorch": {
      "top1": 0.9208,
      "top5": 0.9956
    },
    "vgg16_int8_onnx": {
      "top1": 0.9096,
      "top5": 0.9928
    },
    "int8_fp32_prediction_agreement": 0.9616,
    "n_test": 2500
  },
  "size_mb": {
    "small_cnn_fp32_pytorch": 1.578261,
    "vgg16_fp32_pytorch": 110.271689,
    "vgg16_fp32_onnx": 110.269023,
    "vgg16_int8_onnx": 27.672997
  },
  "latency_cpu_ms": {
    "small_cnn_fp32_pytorch": {
      "mean_ms": 20.227684099972976,
      "std_ms": 2.9109310543676994,
      "runs": 100
    },
    "vgg16_fp32_pytorch": {
      "mean_ms": 614.9414081900386,
      "std_ms": 122.19852761072133,
      "runs": 100
    },
    "vgg16_fp32_onnx": {
      "mean_ms": 472.25198578000345,
      "std_ms": 97.02815988491248,
      "runs": 100
    },
    "vgg16_int8_onnx": {
      "mean_ms": 290.486609400009,
      "std_ms": 63.10342148946426,
      "runs": 100
    }
  },
  "calibration": "200 train images (20 per class), MinMax, QDQ, per-channe